Generation of questions and réponses with llama 3.1 : 8b

In [14]:
%pip install ollama
import ollama


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.3.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Test if Ollama right 

In [15]:
response = ollama.chat(
    model="llama3.1:8b",
    messages=[
        {
            "role": "user",
            "content": "Say hello in one sentence."
        }
    ]
)

print(response["message"]["content"])

Hello, how are you today?


Generation of questions/answers

In [16]:
from select_chunks import selection
# Test for 1 chunks 
chunk = selection[0]

prompt = f"""
You are given a technical excerpt from a 3GPP specification.

Generate one precise technical question whose answer is explicitly contained in the excerpt.

Then generate the reference answer using only the excerpt.

Do not use outside knowledge.
Do not invent information.

Return exactly:

Question: ...
Ground truth: ...

Context:
{chunk["document"]}
"""

response = ollama.chat(
    model="llama3.1:8b",
    messages=[
        {
            "role": "user",
            "content": prompt
        }
    ]
)

print(response["message"]["content"])

Here are the question and answer based on the provided excerpt:

**Question:** For an E-UTRA TDD system operating on Band 54, what is the value of f1, f5?

**Ground Truth:** Mid


In [20]:
#For 20 chunks 
import ollama

qa_results = []

for i, chunk in enumerate(selection, start=1):

    prompt = f"""
You are given a technical excerpt from a 3GPP specification.

Generate one precise technical question whose answer is explicitly contained in the excerpt.

Then generate the reference answer using only the excerpt.

Do not use outside knowledge.
Do not invent information.

Return exactly:

Question: ...
Ground truth: ...

Context:
{chunk["document"]}
"""

    response = ollama.chat(
        model="llama3.1:8b",
        messages=[
            {
                "role": "user",
                "content": prompt
            }
        ]
    )

    generated_text = response["message"]["content"]

    qa_results.append({
        "id": chunk["id"],
        "spec": chunk["metadata"].get("spec"),
        "section": chunk["metadata"].get("section"),
        "source_chunk": chunk["document"],
        "generated_qa": generated_text
    })

    print(f"Chunk {i}/{len(selection)} terminé")

Chunk 1/20 terminé
Chunk 2/20 terminé
Chunk 3/20 terminé
Chunk 4/20 terminé


KeyboardInterrupt: 

In [18]:
for item in qa_results:
    print("=" * 80)
    print("SPEC :", item["spec"])
    print("SECTION :", item["section"])
    print(item["generated_qa"])
    print()

SPEC : 36508-i20_s05-s08
SECTION : 6.2.3.1	Test frequencies for signalling test
Here is the technical question and answer based on the given excerpt:

**Question:** What is the value of f1, f5 for E-UTRA operating band 53?

**Ground Truth:** The value of f1, f5 for E-UTRA operating band 53 is "Low".

SPEC : 4
SECTION : Unknown Section
Question: What is the value of "EvMin" in dBm for QRXL in the given table?

Ground truth: -140

SPEC : 38521-1-i00_s0605
SECTION : 4.3.6.1.1.2	ensuring UL/SUL indicator in Table 4.3.6.1.1.2-1 with
Here is the question and answer:

**Question:** What types of UE are covered by this test case?

**Ground truth:** This test case applies to all types of NR power class 1.5, power class 2 and power class 3 UE.

Note: The question is based on the information explicitly stated in the excerpt, and the answer is a direct quote from the excerpt.

SPEC : 26926-i00
SECTION : 10	XR Distributed Computing
Question: What is the average bitrate for a 3GPP video encoded with

In [19]:
import json

with open("qa_dataset.json", "w", encoding="utf-8") as f:
    json.dump(
        qa_results,
        f,
        ensure_ascii=False,
        indent=4
    )

In [21]:
from select_chunks import selection

print(len(selection))

20
